In [ ]:
## S5: Stratified Split + prepare Pipeline
# Run order: 01_eda.ipynb -> 02_feature_engineering.ipynb -> 03_prepare_dataset.ipynb -> 04_baseline_models.ipynb -> 05_extension_smote_xgboost.ipynb -> 06_shap_cost_sensitive.ipynb

# [18] Chuẩn bị dữ liệu đầu vào cho mô hình bằng cách tải tập dữ liệu đã engineer từ notebook trước.
# Mục tiêu: đảm bảo dữ liệu đã có các feature phù hợp trước khi chia train/test.
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'

engineered_path = DATA_DIR / 'engineered_features.pkl'
processed_path = DATA_DIR / 'processed_split.pkl'
if not engineered_path.exists():
    raise FileNotFoundError(
        'Missing ../data/engineered_features.pkl. Run 02_feature_engineering.ipynb first.'
    )

# [19] Tải bộ dữ liệu đã xử lý từ file pickle để tránh phải chạy lại bước engineering.
df = joblib.load(engineered_path)
print(f'Loaded engineered data: {df.shape}')

# [20] Tách tập X (features) và y (target) cho bài toán phân lớp gian lận.
# isFraud là nhãn cần dự đoán, còn isFlaggedFraud được bỏ vì đây là cột phụ trợ và không dùng làm target.
X = df.drop(columns=['isFraud'])
y = df['isFraud']
X = X.drop(columns=['isFlaggedFraud'])
print(X.shape, y.shape)
print(y.value_counts(normalize=True))

# [21] Bước 1: tách test 20% từ toàn bộ dữ liệu.
# Test set được khóa ngay từ bước này và không được dùng cho việc chọn mô hình.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

# [22] Bước 2: tách validation 20% từ phần train_full còn lại.
# Kết quả cuối: train 64%, validation 16%, test 20%.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42,
)

print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())
print("Test fraud rate:", y_test.mean())

# [23] Chuẩn hóa dữ liệu.
# Scaler chỉ học thống kê từ train để tránh data leakage.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# [24] Tổng kết kích thước và tỷ lệ fraud của ba tập.
split_summary = pd.DataFrame({
    "Tập dữ liệu": ["Train", "Validation", "Test"],
    "Số dòng": [len(X_train), len(X_val), len(X_test)],
    "Tỷ lệ dữ liệu": [
        len(X_train) / len(X),
        len(X_val) / len(X),
        len(X_test) / len(X),
    ],
    "Số thuộc tính": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1],
    ],
    "Tỷ lệ fraud": [
        y_train.mean(),
        y_val.mean(),
        y_test.mean(),
    ],
})

print("\nTổng kết chia tập dữ liệu:")
print(
    split_summary.to_string(
        index=False,
        formatters={
            "Số dòng": "{:,.0f}".format,
            "Tỷ lệ dữ liệu": "{:.2%}".format,
            "Tỷ lệ fraud": "{:.4%}".format,
        },
    )
)

# [25] Lưu các tập dữ liệu và scaler cho notebook phía sau.
joblib.dump({
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "X_train_scaled": X_train_scaled,
    "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_test_scaled,
    "scaler": scaler,
}, processed_path)

print(f"\nSaved processed split to {processed_path}")

Loaded engineered data: (6362620, 17)
(6362620, 15) (6362620,)
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


Train fraud rate: 0.0012907418642005967
Test fraud rate: 0.0012911347840983745


Saved processed split to D:\GitHub\fraud-detection-thesis\data\processed_split.pkl
